# azure_ai_agentserver_responses smoke test

Tests the `azure.ai.agentserver.responses` package — Azure AI Agent Server Responses client library.

In [ ]:
# Example 1: Verify top-level package metadata & public surface.
import azure.ai.agentserver.responses as responses_pkg

version = getattr(responses_pkg, '__version__', None)
print(f"version: {version}")

public_names = [n for n in dir(responses_pkg) if not n.startswith('_')]
print(f"public attribute count: {len(public_names)}")
print(f"first 20 public attributes: {public_names[:20]}")

assert hasattr(responses_pkg, '__path__'), "expected azure.ai.agentserver.responses to be a package"
assert {"CreateResponse", "ResponseEventStream"} <= set(public_names), public_names

stream = responses_pkg.ResponseEventStream(
    response_id="resp_notebook_1", request=responses_pkg.CreateResponse(model="m-1", input="hello"))
events = [stream.emit_created(), stream.emit_in_progress(), *stream.output_item_message("hi there"), stream.emit_completed()]
print(f"event types {[event['type'] for event in events]}")
assert [event["type"] for event in events] == [
    "response.created", "response.in_progress", "response.output_item.added", "response.content_part.added",
    "response.output_text.delta", "response.output_text.done", "response.content_part.done",
    "response.output_item.done", "response.completed"], [event["type"] for event in events]
assert [event["sequence_number"] for event in events] == list(range(9))
response = stream.response
assert (response["id"], response["status"], response["model"]) == ("resp_notebook_1", "completed", "m-1")
assert response["output"][0]["content"][0]["text"] == "hi there", response["output"]

In [ ]:
# Example 2: Confirm the azure namespace package layout is intact.
import azure
import azure.ai
import azure.ai.agentserver
import azure.ai.agentserver.responses

for ns in (azure, azure.ai, azure.ai.agentserver, azure.ai.agentserver.responses):
    print(f"{ns.__name__}: path={list(ns.__path__)[:1]}")
    assert hasattr(ns, '__path__'), f"{ns.__name__} should be a namespace/regular package"

# The leaf module should be importable directly via its dotted name.
import importlib
mod = importlib.import_module('azure.ai.agentserver.responses')
assert mod is azure.ai.agentserver.responses

In [ ]:
# Example 3: Inspect runtime dependency surfaces (azure_core, isodate, aiohttp).
# These are declared deps; verify they import inside the same environment.
import azure.core
import isodate
import aiohttp

print(f"azure.core: {azure.core.__name__}")
print(f"isodate: {getattr(isodate, '__version__', 'unknown')}")
print(f"aiohttp: {aiohttp.__version__}")

# isodate is used by azure SDKs for ISO 8601 durations — sanity-check it.
from datetime import timedelta
duration = isodate.parse_duration('PT1H30M')
assert duration == timedelta(hours=1, minutes=30), f"unexpected duration: {duration}"

In [ ]:
# Example 4: Locate any exported classes / public callables.
# Azure SDK client libraries typically expose a *Client class plus models/operations submodules.
import inspect
import azure.ai.agentserver.responses as responses_pkg

classes = []
callables = []
for name in dir(responses_pkg):
    if name.startswith('_'):
        continue
    obj = getattr(responses_pkg, name)
    if inspect.isclass(obj):
        classes.append(name)
    elif callable(obj):
        callables.append(name)

print(f"classes ({len(classes)}): {classes[:15]}")
print(f"other callables ({len(callables)}): {callables[:15]}")

# At minimum, the package should expose *something* in its public API.
public_names = [n for n in dir(responses_pkg) if not n.startswith('_')]
assert len(public_names) >= 1, "expected at least one public name in azure.ai.agentserver.responses"

In [ ]:
# Example 5: Import the conventional Azure SDK submodules (models / aio / _version); this package ships all three.
import importlib
import importlib.metadata

candidates = [
    'azure.ai.agentserver.responses.models',
    'azure.ai.agentserver.responses.aio',
    'azure.ai.agentserver.responses._version',
]

modules = {}
for name in candidates:
    modules[name] = importlib.import_module(name)
    print(f"OK   {name}")

# `_version.VERSION` is the distribution's own version string.
version = modules['azure.ai.agentserver.responses._version'].VERSION
print(f"VERSION {version}")
assert version == importlib.metadata.version('azure-ai-agentserver-responses'), version